# DVC부터 release manifest까지 하나의 모델 기록 추적하기

## 이번 질문

Git은 코드 변경을, DVC는 데이터와 파이프라인 상태를, MLflow Run은 한 번의 학습 조건과 결과를, 모델 묶음은 실행 파일을 기록합니다. 이 노트북은 흩어진 네 기록이 `release-manifest.json`에서 Candidate B 하나로 이어지는지 순서대로 확인합니다.

공식 Run은 과거 release evidence이고 학생 Run은 같은 구조를 직접 관찰하기 위한 새 개발 실행입니다. 공식 Run과 학생 Run은 서로 다른 실행이며, 학생 Run으로 공식 승인이나 봉인 평가를 바꾸지 않습니다.

## 먼저 예상

Candidate B의 모델 Run ID, 최종 평가 Run ID, 모델 파일 SHA-256이 같은 값인지 먼저 예상합니다. Git commit, DVC lock SHA-256, 데이터 파일 SHA-256, MLflow Run ID, 모델 파일 SHA-256 가운데 서로 대신할 수 있는 식별값은 없습니다.

여기서 `v2`는 이 과정이 정한 **데이터 분할 revision 이름**입니다. DVC가 자동 발급한 commit이나 experiment ID가 아닙니다. DVC의 구체적인 재현 상태는 `dvc.lock` 파일의 SHA-256과 stage별 dependency/output hash로 따로 식별합니다.

## 실행과 관측

### 1. 연결에 필요한 선언을 한 번에 연다

먼저 DVC 분할, 모델 생성, release freeze, 최종 승인, 직렬화 검증 문서를 읽습니다. 각 파일의 역할은 다르며, 뒤 셀에서 Candidate B를 공통 열쇠로 연결합니다. 이 V2 기록은 역사적 migration evidence이므로 현재의 정상 lifecycle과 동일하다고 가정하지 않습니다.

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd
import yaml

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir()
)
paths = {
    "current_dvc_lock": ROOT / "dvc.lock",
    "dvc_split": ROOT / "docs/evidence/data-v2/split-revision.json",
    "model_bootstrap": ROOT / "docs/evidence/model-v2/model-bootstrap.json",
    "release_freeze": ROOT / "docs/evidence/model-v2/release-freeze.json",
    "canonical": ROOT / "docs/evidence/model-v2/canonical-benchmark.json",
    "release_manifest": ROOT / "docs/evidence/model-v2/release-manifest.json",
    "bundle_verification": ROOT
    / "docs/evidence/model-v2/serialized-bundle-verification.json",
    "profiles": ROOT / "configs/model-v2/profiles.yaml",
}
documents = {
    name: (
        yaml.safe_load(path.read_text(encoding="utf-8"))
        if path.suffix in {".yaml", ".yml", ".lock"}
        else json.loads(path.read_text(encoding="utf-8"))
    )
    for name, path in paths.items()
}
pd.DataFrame(
    {
        "역할": [
            "현재 repository DVC stage의 구체적인 재현 상태",
            "V2 분할 revision과 역할별 데이터 지문",
            "역사적 모델 Run과 최초 bundle 지문",
            "역사적 V2 freeze migration 상태",
            "봉인 평가 결과",
            "승인 profile과 최종 Run 연결",
            "bundle 직렬화 검증",
            "모델 종류와 임계값 선언",
        ],
        "경로": [path.relative_to(ROOT).as_posix() for path in paths.values()],
    },
    index=list(paths),
)

### 2. 확인된 연결과 역사적 한계를 분리한다

Git commit은 코드를, DVC lock과 dataset SHA-256은 데이터 재현 상태를, MLflow Run ID는 실행을, model/metadata SHA-256은 파일 내용을 식별합니다. `release-manifest.json`은 이 값을 새 식별자로 합치지 않고 승인된 Candidate B와 연결합니다.

다만 V2는 현재 lifecycle 도입 전에 만들어진 역사적 evidence입니다. 모델 생성 당시 lock, V2 분할 선언에 적힌 lock, 현재 repository lock의 SHA-256이 서로 다르고 원본 frozen lock blob도 남아 있지 않습니다. `model-bootstrap.json`에는 당시 Git worktree가 dirty였다고 기록돼 있습니다. 따라서 이 노트북이 확인하는 것은 **동일한 train/valid/test 파일 지문, 공식 평가 수치, bundle 지문과 Run ID의 사후 reconciliation**입니다. 과거 DVC stage 전체를 원본 lock에서 다시 실행해 같은 Run을 재현했다고 설명하면 안 됩니다.

In [ ]:
split_revision = documents["dvc_split"]
bootstrap = documents["model_bootstrap"]
release_freeze = documents["release_freeze"]
canonical = documents["canonical"]
release_manifest = documents["release_manifest"]
candidate_bundle = bootstrap["bundles"]["candidate-b"]
sealed_role = split_revision["role_datasets"]["test"]

current_dvc_lock_sha256 = hashlib.sha256(paths["current_dvc_lock"].read_bytes()).hexdigest()
split_dvc_lock_sha256 = split_revision["configuration"]["dvc_lock_sha256"]
historical_dvc_lock_sha256 = bootstrap["provenance"]["dvc_lock_revision"]
historical_reconciliation = release_manifest["historical_reconciliation"]
history_boundary = pd.DataFrame(
    [
        {
            "기록": "현재 repository dvc.lock",
            "SHA-256 또는 상태": current_dvc_lock_sha256,
            "설명 가능한 범위": "현재 코드와 데이터 pipeline의 재현 상태",
        },
        {
            "기록": "V2 split 선언 당시 dvc.lock",
            "SHA-256 또는 상태": split_dvc_lock_sha256,
            "설명 가능한 범위": "split-revision.json이 기록한 분할 생성 상태",
        },
        {
            "기록": "역사적 모델 생성 당시 dvc.lock",
            "SHA-256 또는 상태": historical_dvc_lock_sha256,
            "설명 가능한 범위": "model-bootstrap.json에 남은 지문만 확인 가능",
        },
        {
            "기록": "원본 frozen lock snapshot",
            "SHA-256 또는 상태": historical_reconciliation[
                "frozen_dvc_lock_snapshot_available"
            ],
            "설명 가능한 범위": "원본 blob 없음: 과거 stage 전체 재현 주장 불가",
        },
        {
            "기록": "역사적 모델 생성 Git 상태",
            "SHA-256 또는 상태": bootstrap["provenance"]["git_worktree_dirty"],
            "설명 가능한 범위": "dirty 기록: clean source 재현 주장 불가",
        },
    ]
).set_index("기록")
display(history_boundary)

# 역사적 한계를 인정한 뒤, 실제로 대조 가능한 식별값만 연결한다.
official_model_run = candidate_bundle["mlflow_run_id"]
official_final_run = release_manifest["approved_model"]["final_mlflow_run_id"]
official_chain = pd.DataFrame(
    [
        {
            "기록": "historical Git code revision",
            "값": bootstrap["provenance"]["git_commit"],
            "의미": "모델 생성 기록에 남은 code commit",
            "근거": "model-bootstrap.json",
        },
        {
            "기록": "course data revision",
            "값": f"{split_revision['revision']} / sealed role {sealed_role['rows']} rows",
            "의미": "과정이 정의한 데이터 분할 이름과 역할",
            "근거": "split-revision.json",
        },
        {
            "기록": "sealed dataset SHA-256",
            "값": sealed_role["sha256"],
            "의미": "사후 reconciliation으로 확인한 봉인 데이터 파일",
            "근거": "split-revision.json / release-manifest.json",
        },
        {
            "기록": "historical model Run ID",
            "값": official_model_run,
            "의미": "Candidate B 모델 생성 기록의 실행 ID",
            "근거": "model-bootstrap.json",
        },
        {
            "기록": "model.joblib SHA-256",
            "값": candidate_bundle["model_sha256"],
            "의미": "검증된 실행 모델 파일 내용",
            "근거": "model-bootstrap.json",
        },
        {
            "기록": "metadata.json SHA-256",
            "값": candidate_bundle["metadata_sha256"],
            "의미": "검증된 외부 설명 파일 내용",
            "근거": "model-bootstrap.json",
        },
        {
            "기록": "historical final Run ID",
            "값": official_final_run,
            "의미": "봉인 평가 기록의 실행 ID",
            "근거": "release-manifest.json",
        },
        {
            "기록": "approved profile",
            "값": release_manifest["approved_profile"],
            "의미": "최종 승인된 모델 종류",
            "근거": "release-manifest.json",
        },
    ]
).set_index("기록")
official_chain

### 3. baseline과 Candidate B의 실행 파일 묶음을 구분한다

하나의 모델 묶음은 `model.joblib`과 `metadata.json` 두 파일입니다. 두 파일은 서로 다른 SHA-256을 가지며, 공통 입력 항목 규약의 SHA-256도 별도로 유지합니다. 지문이 다르면 이름이 같아도 다른 실행 파일 묶음으로 조사합니다.

In [ ]:
bundle_verification = documents["bundle_verification"]
feature_contract_sha256 = release_manifest["approved_model"][
    "feature_contract_sha256"
]
bundle_rows = []
for profile in ("baseline", "candidate-b"):
    verified = bundle_verification["profiles"][profile]
    bundle_rows.append(
        {
            "profile": profile,
            "model.joblib SHA-256": verified["model_sha256"],
            "metadata.json SHA-256": verified["metadata_sha256"],
            "공통 feature contract SHA-256": feature_contract_sha256,
            "canonical metrics 일치": verified["matches_canonical_metrics"],
        }
    )
bundle_comparison = pd.DataFrame(bundle_rows).set_index("profile")
bundle_comparison

### 4. 역사적 Candidate B 기록을 여러 문서에서 재구성한다

아래 표는 실제 MLflow Run export가 아니라 `split-revision.json`, 현재 profile YAML, `development-benchmark.json`, `model-bootstrap.json`을 profile 이름으로 조합한 **cross-document reconstruction**입니다. 학습 2,900건과 검증 600건, Random Forest와 임계값 0.35, 검증 Recall과 PR-AUC, 모델·설명 파일의 기대 연결을 보여줍니다.

Run `31b50eb...`의 원본 dataset input, parameter, metric, artifact를 Run ID로 export한 문서는 남아 있지 않습니다. 따라서 이 표를 “MLflow 서버에서 공식 Run을 직접 검증한 결과”라고 설명하지 않습니다.

In [ ]:
development_path = ROOT / "docs/evidence/model-v2/development-benchmark.json"
development = json.loads(development_path.read_text(encoding="utf-8"))
candidate_profile = next(
    item for item in documents["profiles"]["profiles"] if item["name"] == "candidate-b"
)
candidate_evaluation = next(
    item for item in development["profiles"] if item["profile"] == "candidate-b"
)
historical_run_reconstruction = pd.DataFrame(
    {
        "값": [
            split_revision["role_datasets"]["train"]["rows"],
            split_revision["role_datasets"]["valid"]["rows"],
            candidate_profile["kind"],
            candidate_profile["threshold"],
            candidate_evaluation["metrics"]["recall"],
            candidate_evaluation["metrics"]["pr_auc"],
            official_model_run,
            candidate_bundle["model_path"],
            candidate_bundle["metadata_path"],
        ],
        "원본": [
            "split-revision.json",
            "split-revision.json",
            "configs/model-v2/profiles.yaml",
            "configs/model-v2/profiles.yaml",
            "development-benchmark.json",
            "development-benchmark.json",
            "model-bootstrap.json",
            "model-bootstrap.json (bundle path, Run artifact 확인 아님)",
            "model-bootstrap.json (bundle path, Run artifact 확인 아님)",
        ],
    },
    index=[
        "train rows",
        "valid rows",
        "model kind",
        "threshold",
        "valid recall",
        "valid PR-AUC",
        "historical model Run ID",
        "recorded model bundle path",
        "recorded metadata bundle path",
    ],
)
historical_run_reconstruction

### 5. 학생 MLP Run에서 조건, iteration 곡선과 생성 파일을 함께 조회한다

`labs/run/log_development.py`는 `split-revision.json`의 revision, 경로, 행 수와 SHA-256을 먼저 검증한 뒤 그 **동일한 `train.csv`와 `valid.csv`**를 학습 입력과 MLflow dataset input으로 사용합니다. `test`와 `operational`은 열지 않습니다. 교실 MLflow가 준비되면 experiment `student-development-tracking`에 sklearn `MLPClassifier`(`candidate-c`)의 dataset input, 모델 설정, iteration별 `train.loss`/`valid.*` 지표, `bundle/model.joblib`, `bundle/metadata.json`, MLflow logged model을 한 Run으로 기록합니다.

이 실행은 구조를 직접 관찰하기 위한 학생 Run입니다. 공식 승인 모델은 여전히 Candidate B Random Forest입니다. 학생 MLP Run ID는 반드시 새 값이며, 이 Run으로 공식 승인이나 봉인 평가를 바꾸지 않습니다. 데이터가 준비되지 않았거나 `AIQA_MLFLOW_TRACKING_URI`가 없으면 역사적 연결 표까지만 확인하고 막힌 조건을 그대로 표시합니다.

In [ ]:
import os
import subprocess

required_development_files = [
    ROOT / "docs/evidence/data-v2/split-revision.json",
    ROOT / "data/splits-v2/train.csv",
    ROOT / "data/splits-v2/valid.csv",
]
missing_development_files = [
    path.relative_to(ROOT).as_posix()
    for path in required_development_files
    if not path.is_file()
]
if missing_development_files:
    student_result = {
        "status": "DATA_NOT_PREPARED",
        "next_action": "uv run python labs/run/prepare_data.py",
        "missing": missing_development_files,
        "student_run_id": None,
    }
else:
    completed = subprocess.run(
        ["uv", "run", "python", "labs/run/log_development.py"],
        cwd=ROOT,
        env=os.environ.copy(),
        check=True,
        capture_output=True,
        text=True,
    )
    student_result = json.loads(completed.stdout)

pd.DataFrame(
    {"값": list(student_result.values())},
    index=list(student_result),
)

In [ ]:
if student_result["status"] == "LOGGED":
    import mlflow
    from mlflow import MlflowClient
    from mlflow.models import get_model_info

    tracking_uri = os.environ["AIQA_MLFLOW_TRACKING_URI"].rstrip("/")
    mlflow.set_tracking_uri(tracking_uri)
    client = MlflowClient(tracking_uri=tracking_uri)
    student_run = client.get_run(student_result["student_run_id"])
    dataset_inputs = list(getattr(student_run.inputs, "dataset_inputs", []))
    artifact_paths = [item.path for item in client.list_artifacts(student_run.info.run_id)]
    logged_models = list(
        client.search_logged_models(
            experiment_ids=[student_run.info.experiment_id],
            filter_string=f"source_run_id = '{student_run.info.run_id}'",
        )
    )
    logged_model_contracts = {
        model.model_id: get_model_info(f"models:/{model.model_id}")
        for model in logged_models
    }
    live_run_summary = pd.DataFrame(
        {
            "값": [
                student_run.info.run_id,
                student_run.data.tags.get("aiqa.profile"),
                student_run.data.params.get("model_kind"),
                student_run.data.params.get("threshold"),
                student_run.data.metrics.get("valid.recall"),
                student_run.data.metrics.get("valid.pr_auc"),
                len(dataset_inputs),
                ", ".join(artifact_paths),
                len(logged_models),
                student_result["bundle_model_sha256"],
                student_result["bundle_metadata_sha256"],
            ]
        },
        index=[
            "student_run_id",
            "profile tag",
            "model_kind parameter",
            "threshold parameter",
            "valid.recall metric",
            "valid.pr_auc metric",
            "dataset input 수",
            "Run root artifacts",
            "MLflow logged model 수",
            "bundle_model_sha256",
            "bundle_metadata_sha256",
        ],
    )
    live_dataset_summary = pd.DataFrame(
        [
            {
                "name": item.dataset.name,
                "context": item.tags[0].value if item.tags else "",
                "MLflow digest": item.dataset.digest,
                "source_type": item.dataset.source_type,
                "source": item.dataset.source,
            }
            for item in dataset_inputs
        ]
    )
    live_logged_model_summary = pd.DataFrame(
        [
            {
                "model_id": model.model_id,
                "name": model.name,
                "source_run_id": model.source_run_id,
                "status": str(model.status),
                "model_uri": logged_model_contracts[model.model_id].model_uri,
                "signature input 수": len(
                    logged_model_contracts[model.model_id].signature.inputs
                ),
                "input example": logged_model_contracts[
                    model.model_id
                ].saved_input_example_info["artifact_path"],
            }
            for model in logged_models
        ]
    )
    display(live_run_summary)
    display(live_dataset_summary)
    display(live_logged_model_summary)
else:
    dataset_inputs = []
    artifact_paths = []
    logged_models = []
    live_run_summary = pd.DataFrame(
        {
            "값": [
                student_result["status"],
                student_result.get("next_action", "교실 MLflow 준비를 확인합니다."),
            ]
        },
        index=["학생 Run 상태", "다음 확인"],
    )
    display(live_run_summary)

### 6. MLflow 화면과 실제 코드를 한 항목씩 연결한다

MLflow의 한 Run 화면에는 서로 다른 계약이 함께 보입니다. Tags는 검색·설명, Parameters는 실행 조건, Metrics는 측정 결과, Datasets는 실제 입력 identity, Artifacts는 Run에 첨부한 일반 파일, Logged Models는 signature와 flavor를 가진 MLflow model입니다.

`bundle/model.joblib`과 `bundle/metadata.json`은 `log_artifacts()`가 올린 **일반 Run artifact**입니다. `mlflow.sklearn.log_model()`이 만든 **Logged Model**은 별도 model ID, URI, signature, input example과 artifact location을 가집니다. 둘은 같은 fitted pipeline에서 만들어져도 역할과 조회 화면이 다릅니다.

현재 코드는 `register_model()`이나 `registered_model_name`을 호출하지 않으므로 Model Registry의 registered model/version/alias를 만들지는 않습니다. 승인 기준은 mutable alias가 아니라 bundle SHA-256과 `release-manifest.json`입니다.

Datasets 화면의 32자리 MLflow digest는 `mlflow_dataset_digest()`가 원본 CSV SHA-256의 앞 32자를 MLflow dataset 필드에 맞춰 전달한 축약 identity입니다. 원본 CSV byte의 64자리 SHA-256은 Parameters의 `train_data_hash`와 `valid_data_hash`에 온전히 기록합니다. 축약값은 앞부분을 대조하는 UI 보조값이며 원본 무결성 검증을 대신하지 않습니다. 아래 표를 왼쪽 MLflow 화면, 오른쪽 구현 코드 순서로 읽습니다.

In [ ]:
ui_code_map = pd.DataFrame(
    [
        {
            "MLflow 화면": "Tags",
            "기록 API": "mlflow.set_tags",
            "이 실습의 값": "profile, candidate_id, evaluation_role, not_official_evidence",
            "설명 질문": "이 Run을 어떤 목적으로 검색하고 구분하는가",
        },
        {
            "MLflow 화면": "Parameters",
            "기록 API": "mlflow.log_params",
            "이 실습의 값": "model_kind, threshold, model.*, Git/DVC/config hashes",
            "설명 질문": "어떤 조건과 provenance로 실행했는가",
        },
        {
            "MLflow 화면": "Metrics",
            "기록 API": "mlflow.log_metric(..., step=iteration)",
            "이 실습의 값": "train.loss와 valid.* 곡선, 최종 FN/CI",
            "설명 질문": "MLP iteration마다 검증 결과가 어떻게 변했는가",
        },
        {
            "MLflow 화면": "Datasets",
            "기록 API": "mlflow_dataset_digest + mlflow.data.from_pandas + mlflow.log_input",
            "이 실습의 값": "train/valid name, context, source, SHA-256 앞 32자",
            "설명 질문": "실제 입력과 축약 dataset identity는 무엇인가",
        },
        {
            "MLflow 화면": "Artifacts / bundle",
            "기록 API": "mlflow.log_artifacts",
            "이 실습의 값": "bundle/model.joblib, bundle/metadata.json",
            "설명 질문": "외부 시스템도 검증할 수 있는 실행 파일 묶음은 무엇인가",
        },
        {
            "MLflow 화면": "Logged Models",
            "기록 API": "mlflow.sklearn.log_model",
            "이 실습의 값": "model ID, signature, input example, sklearn flavor",
            "설명 질문": "MLflow가 load할 수 있는 model contract는 무엇인가",
        },
    ]
).set_index("MLflow 화면")
display(ui_code_map)

code_map = pd.DataFrame(
    [
        {
            "순서": 1,
            "경로와 함수": "dvc.yaml / dvc.lock / split-revision.json",
            "읽을 질문": "현재 DVC stage 상태와 과정의 V2 데이터 revision은 어떻게 다른가",
        },
        {
            "순서": 2,
            "경로와 함수": "labs/run/development.yaml",
            "읽을 질문": "학생 Run이 사용할 profile, role, 입력과 experiment를 어디서 고정하는가",
        },
        {
            "순서": 3,
            "경로와 함수": "labs/run/log_development.py::verify_development_lineage / run_student_development",
            "읽을 질문": "같은 train/valid 파일을 검증·학습·기록하도록 어떻게 강제하는가",
        },
        {
            "순서": 4,
            "경로와 함수": "packages/aiqa_model/adapters/mlflow/runtime.py::configure_tracking",
            "읽을 질문": "교실 server와 experiment를 어떻게 선택하는가",
        },
        {
            "순서": 5,
            "경로와 함수": "packages/aiqa_model/adapters/mlflow/model.py::MlflowModelTracker.record / datasets.py::mlflow_dataset_digest",
            "읽을 질문": "UI 값을 어디서 기록하며 64자리 SHA-256을 dataset field에 어떻게 연결하는가",
        },
        {
            "순서": 6,
            "경로와 함수": "packages/aiqa_model/adapters/bundles/joblib.py::persist_model_bundle",
            "읽을 질문": "model.joblib과 metadata.json을 어떤 무결성 계약으로 만드는가",
        },
        {
            "순서": 7,
            "경로와 함수": "apps/model_trainer/application/bundles.py::bootstrap_models",
            "읽을 질문": "새 revision에서 train/valid bundle과 model Run을 언제 고정하는가",
        },
        {
            "순서": 8,
            "경로와 함수": "apps/model_trainer/application/finalization.py::run_final",
            "읽을 질문": "freeze 뒤 sealed test와 final Run을 어떻게 한 번만 실행하는가",
        },
        {
            "순서": 9,
            "경로와 함수": "apps/model_trainer/adapters/release_provenance.py::write_release_freeze / write_release_manifest",
            "읽을 질문": "승인 전 freeze와 승인 후 manifest가 왜 별도 문서인가",
        },
        {
            "순서": 10,
            "경로와 함수": "docs/evidence/model-v2/README.md",
            "읽을 질문": "development, freeze, sealed evaluation, decision과 publish evidence의 전체 순서는 무엇인가",
        },
        {
            "순서": 11,
            "경로와 함수": "labs/appendix/09_dvc_basics.ipynb / 13_mlflow_basics.ipynb",
            "읽을 질문": "DVC와 MLflow 개별 문법을 작은 격리 예제로 다시 확인할 때",
        },
    ]
).set_index("순서")
code_map

## 해석과 기록

하나의 모델 기록은 하나의 만능 ID가 아니라 역할이 다른 식별값의 연결입니다. Git commit은 코드 상태, DVC lock과 dataset SHA-256은 데이터 재현 상태, MLflow Run ID는 실행, model/metadata SHA-256은 파일 내용을 가리킵니다. `release-manifest.json`은 승인된 profile, 모델 Run, 최종 Run과 파일 지문을 연결합니다.

역사적 V2는 원본 frozen DVC lock과 clean worktree를 복원할 수 없는 migration evidence입니다. 확인 가능한 dataset, canonical metric과 bundle 지문을 reconciliation했지만, 현재의 정상 lifecycle을 과거에 그대로 수행했다고 말하지 않습니다.

새 revision을 도입할 때의 정상 순서는 다음과 같습니다.

1. versioned YAML과 DVC pipeline으로 train/valid/test 역할과 입력 지문을 고정합니다.
2. train/valid만 사용해 후보를 개발하고 model Run, bundle과 provenance를 기록합니다.
3. clean Git commit, DVC/data identity, config와 bundle 지문을 `release-freeze.json`에 고정합니다.
4. freeze를 검토한 뒤 sealed test를 한 번만 열어 final Run과 canonical evidence를 만듭니다.
5. 승인된 profile, 두 Run ID와 모든 지문을 `release-manifest.json`에 연결합니다.
6. 배포에서는 manifest의 model SHA-256과 immutable image digest를 실제 runtime identity와 대조합니다.

학생 Candidate B Run은 이 구조를 직접 조회하기 위한 개발 실행입니다. 공식 Run과 숫자나 model SHA-256이 같아도 공식 evidence가 되지 않습니다. Run ID는 새 실행마다 달라지지만 deterministic serialization이면 model SHA-256은 같을 수도 있습니다.

## 결과 점검

In [ ]:
assert split_revision["revision"] == "v2"
assert sealed_role["rows"] == 400
assert len(
    {
        current_dvc_lock_sha256,
        split_dvc_lock_sha256,
        historical_dvc_lock_sha256,
    }
) == 3
assert historical_reconciliation["frozen_dvc_lock_snapshot_available"] is False
assert bootstrap["provenance"]["git_worktree_dirty"] == "true"
assert release_freeze["schema_version"] == 2
assert canonical["sealed_test"]["dataset_sha256"] == sealed_role["sha256"]
assert release_manifest["approved_profile"] == "candidate-b"
assert release_manifest["approved_model"]["model_mlflow_run_id"] == official_model_run
assert official_model_run != official_final_run
assert release_manifest["model_bundles"]["candidate-b/model.joblib"] == candidate_bundle[
    "model_sha256"
]
assert release_manifest["model_bundles"]["candidate-b/metadata.json"] == candidate_bundle[
    "metadata_sha256"
]
assert release_freeze["model_bundles"]["candidate-b/model.joblib"] == candidate_bundle[
    "model_sha256"
]
assert release_freeze["sha256"]["feature_contract_path"] == feature_contract_sha256
assert bundle_comparison.loc["baseline", "model.joblib SHA-256"] != bundle_comparison.loc[
    "candidate-b", "model.joblib SHA-256"
]
assert candidate_profile["kind"] == "random_forest"
assert candidate_profile["threshold"] == 0.35
assert student_result["status"] in {
    "LOGGED",
    "MLFLOW_NOT_RUNNING",
    "DATA_NOT_PREPARED",
}
if student_result["status"] == "LOGGED":
    assert student_result["student_run_id"] != official_model_run
    assert student_result["student_run_id"] != official_final_run
    assert student_result["profile_name"] == "candidate-c"
    assert student_result["model_kind"] == "mlp_classifier"
    assert student_result["threshold"] == 0.35
    assert student_result["iteration_count"] == 30
    assert len(dataset_inputs) == 2
    assert "bundle" in artifact_paths
    assert len(logged_models) == 1
    assert logged_models[0].source_run_id == student_result["student_run_id"]
print("역사적 한계와 현재 DVC, MLflow, bundle, manifest 연결을 확인했습니다.")

## 다음 확인

`labs/chapters/ch02/INSTRUCTOR_GUIDE.md`의 teach-back 체크리스트를 사용해 다음 여섯 문장을 직접 설명합니다.

1. `v2`, DVC lock SHA-256과 dataset SHA-256은 왜 다른가.
2. 역사적 V2에서 확인된 연결과 복원할 수 없는 범위는 무엇인가.
3. 학생 Run의 Parameters, Metrics, Datasets, Artifacts, Logged Models는 어느 코드에서 만들어지는가.
4. `model.joblib`, `metadata.json`과 MLflow Logged Model은 왜 모두 필요한가.
5. model Run과 final Run은 왜 다른가.
6. release manifest의 승인이 실제 배포 완료를 의미하지 않는 이유는 무엇인가.

이후 3장에서 `/v1/model`이 가리키는 profile, version과 model SHA-256을 이 기록의 Candidate B 묶음과 대조합니다. DVC 명령 자체가 더 필요하면 `labs/appendix/09_dvc_basics.ipynb`, MLflow API와 model load가 더 필요하면 `labs/appendix/13_mlflow_basics.ipynb`를 해당 항목에서만 참고합니다.